# 📊 **Project Dashboard - RubikCare** 

## 🛠️ **وثيقة التطوير التقنية الشاملة**

---

## 📋 **دليل سريع للمطورين**

### **🔧 عند إنشاء صفحة جديدة:**
```mermaid
graph TD
    A[صفحة جديدة؟] --> B{نوع الصفحة}
    B -->|CRUD مع DbSet| C[BaseFactoryCrudPage]
    B -->|Identity| D[RoleManager/UserManager]
    B -->|علاقات معقدة| E[BaseFactoryCrudPageWithRelations]
    B -->|صفحة بسيطة| F[اتصال مباشر]
```

### **⚠️ قبل أي تعديل:**
1. ✅ تحقق من وجود DbSet في BusinessDbContext
2. ✅ تحقق من تسجيل الخدمة في Program.cs
3. ✅ استخدم DbContextFactory بدلاً من DbContext مباشر
4. ✅ تأكد من نظام code-first (لا تعديل يدوي في Migration)

---

## 🏗️ **النظام المعماري الحالي**

### **📁 هيكل المشروع (للتنقل السريع):**
```
Rubikcare.Web/
├── 📂 Data/ (مهم!)
│   ├── Models/                    ← كل النماذج هنا
│   │   ├── Medication.cs          ← 17,615 سجل
│   │   ├── DrugForm.cs            ← 56 شكل (جديد)
│   │   ├── MedicationCategory.cs  ← 1,059 تصنيف
│   │   └── [كل النماذج...]
│   │
│   ├── 📂 Services                 ← الخدمات الأساسية
│   │   ├── Core/
│   │   │   └── DbContextFactoryService.cs ⭐ أساسي
│   │   ├── Navigation/
│   │   │   └── DynamicMenuService.cs ← نظام القوائم
│   │   ├── GenericService.cs      ← لـ CRUD العام
│   │   └──📂PSP/               ← خدمات نظام PSP
│   │        ├── IPSPProgramService.cs
│   │        └── PSPProgramService.cs
│   │      
│   │
│   └── Migrations/                ← 40+ ملف Migration
│
├── 📂 Components/Shared/ (مهم للمكونات!)
│   ├── Base/                      ← الصفوف الأساسية
│   │   ├── BaseCrudPage.cs        ← النظام القديم (يبقى)
│   │   ├── BaseFactoryCrudPage.cs ← النظام الجديد ⭐
│   │   └── BaseFactoryCrudPageWithRelations.cs ⭐
│   │
│   └── RubikCare/                 ← المكونات المخصصة
│       ├── RubikSmartTable.razor  ← عرض البيانات
│       ├── RubikDropdown.razor    ← قوائم منسدلة
│       ├── RubikTabs.razor        ← تبويبات
│       ├── GenericModal.razor     ← نوافذ حوارية
│       ├── DataOperationsModal.razor ← Excel
│       ├── SearchBar.razor        ← بحث
│       ├── Pagination.razor       ← ترقيم الصفحات
│       ├── AlertMessage.razor     ← إشعارات
│       ├── RubikButton.razor      ← أزرار مخصصة
│       ├── ImageUploader.razor    ← رفع صور
│       ├── DynamicForm.razor      ← نماذج ديناميكية
│       ├── Rubiklistview.razor    ← عرض قوائم
│       └── ExcelService.cs        ← خدمة Excel
│
├── 📂 Pages/ (هيكل المسارات)
│   ├── Admin/                     ← كل الصفحات الإدارية
│   │   ├── Medical/               ← طبي
│   │   │   ├── Diseases/          ← الأمراض
│   │   │   ├── Medications/       ← الأدوية (65K+)
│   │   │   ├── Specialities/      ← التخصصات
│   │   │   └── LabTests/          ← التحاليل
│   │   │
│   │   ├── General/               ← عام
│   │   │   ├── Countries/         ← دول
│   │   │   ├── Cities/            ← مدن
│   │   │   └── Areas/             ← مناطق
│   │   │
│   │   ├── Users/                 ← مستخدمين
│   │   │   ├── Roles/             ← أدوار
│   │   │   ├── Profiles/          ← ملفات شخصية
│   │   │   └── List/              ← قائمة المستخدمين
│   │   │
│   │   ├── Organizations/         ← مؤسسات (1,241)
│   │   ├── System/                ← نظام
│   │   │   └── Translations/      ← ترجمة (2,375)
│   │   │
│   │   └── PSP/                   ← برامج رعاية
│   │       ├── Programs/          ← برامج
│   │       └── [PSP صفحات أخرى]
│   │
│   └── Layout/                    ← التخطيطات
│       └── InteractiveMenu.razor  ← نظام القوائم ⭐
│
└── 📂 Documentation/              ← الوثائق
    └── Project-Dashboard.md       ← هذه الوثيقة
```

---

## ⚙️ **نظام القوائم الجانبية (InteractiveMenu)**

### **🎯 كيف يعمل:**
```csharp
// DynamicMenuService.cs - المنطق الأساسي
public async Task<List<MenuItem>> GetUserMenusAsync(int userProfileID)
{
    // 1. القائمة الشخصية (USER_BASE) - دائماً
    // 2. قوائم المؤسسات - إذا كان عضو نشط
    // 3. قائمة Admin - إذا كان في مؤسسة روبيك كير
    
    // 🔑 المفتاح: MenuAssignment.OrganizationID nullable
    // فصل تام بين الشخصي والمؤسسي
}
```

### **📁 جداول نظام القوائم:**
```sql
-- 1. SystemMenus: القوائم الرئيسية
-- ID, MenuNameAr, MenuNameEn, MenuType (USER_BASE, ORG_ADMIN, etc.)

-- 2. MenuItems: عناصر القوائم  
-- ID, ParentID, TitleAr, TitleEn, Icon, Url, SystemMenuID

-- 3. MenuAssignments: التخصيصات
-- ID, MenuItemID, UserProfileID, OrganizationID (nullable)
```

### **🔧 حالات الاستخدام:**
```yaml
المستخدم العادي:
  - القائمة الشخصية فقط (USER_BASE)
  - 4 عناصر افتراضية

عضو في مؤسسات:
  - القائمة الشخصية
  + قوائم المؤسسات النشطة
  - زر تبديل بين المؤسسات

عضو في روبيك كير:
  - القائمة الشخصية  
  + قائمة Admin (ADMIN_MENU)
  - إدارة كاملة للنظام
```

### **⚡ الأداء:**
```sql
-- فهارس للبحث السريع:
CREATE INDEX IX_MenuAssignments_UserProfile 
ON MenuAssignments(UserProfileID, OrganizationID);

CREATE INDEX IX_MenuItems_SystemMenu 
ON MenuItems(SystemMenuID, ParentID);
```

---

## 🧩 **مصفوفة المكونات الذكية**

### **✅ المكونات المجربة 100% (13 مكون):**
| المكون | الصفحات المستخدمة | الغرض | الحالة |
|--------|-------------------|-------|--------|
| **GenericModal** | Diseases, AdminMenuManager | نوافذ حوارية | ✅ 100% |
| **DataOperationsModal** | Diseases, AdminMenuManager | استيراد/تصدير Excel | ✅ 100% |
| **SearchBar** | Diseases, AdminMenuManager | بحث متقدم | ✅ 100% |
| **RubikSmartTable** | Diseases, AdminMenuManager | عرض بيانات | ✅ 100% |
| **Pagination** | Diseases, AdminMenuManager | ترقيم صفحات | ✅ 100% |
| **AlertMessage** | Diseases, AdminMenuManager | إشعارات | ✅ 100% |
| **RubikButton** | Diseases, AdminMenuManager | أزرار مخصصة | ✅ 100% |
| **ImageUploader** | UserProfile.razor | رفع صور | ✅ 90% |
| **DynamicForm** | Specialities.razor | نماذج ديناميكية | ✅ 90% |
| **ExcelService** | Diseases, AdminMenuManager | خدمة Excel | ✅ 100% |
| **RubikDropdown** | AdminMenuManager, PSP | قوائم منسدلة | ✅ 100% |
| **RubikTabs** | PSPProgramEdit.razor | تبويبات | ✅ 100% |
| **Rubiklistview** | PSPProgramEdit.razor | عرض قوائم | ✅ 100% |

### **📋 متى تستخدم أي مكون:**
```yaml
عند عرض جدول بيانات:
  - RubikSmartTable (الجدول الرئيسي)
  - SearchBar (فلترة)
  - Pagination (ترقيم)
  - DataOperationsModal (Excel)

عند النماذج والإدخال:
  - GenericModal (لنماذج الإضافة/التعديل)
  - RubikDropdown (لقوائم الاختيار)
  - DynamicForm (لنماذج معقدة)
  - AlertMessage (لتأكيد الحفظ)

عند التبويبات:
  - RubikTabs (لتنظيم المحتوى)
  - Rubiklistview (لعرض قوائم داخل التبويب)
```

---


---

## 🔄 **دليل الترحيل والتطوير** (مصحح)

### **📋 خريطة ترحيل الصفحات - الحالة الحقيقية:**
| # | الصفحة | المسار | النمط الحالي | النظام | الأولوية | حالة Excel |
|---|--------|--------|--------------|----------|----------|------------|
| 1 | **الأمراض** | `/admin/medical/diseases/list` | BaseFactoryCrudPage | ✅ **الجديد** | ⭐⭐⭐⭐⭐ | ✅ يعمل |
| 2 | **إدارة القوائم** | `/admin/system/menus/manager` | BaseFactoryCrudPage | ✅ **الجديد** | ⭐⭐⭐⭐⭐ | ✅ يعمل |
| 3 | **برامج PSP** | `/admin/psp/programs/list` | BaseFactoryCrudPage | ✅ **الجديد** | ⭐⭐⭐ | ✅ يعمل |
| 4 | **التخصصات** | `/admin/medical/specialities/list` | **BaseCrudPage** | ⚡ **مهجن** | ⭐⭐⭐ | ✅ تم الربط |
| 5 | **المناطق/المدن/الدول** | `/admin/general/*` | **BaseCrudPage** | ⚡ **مهجن** | ⭐⭐ | ❓ غير مجرب |
| 6 | **الترجمة** | `/admin/system/translations/list` | **BaseCrudPage** | ⚡ **مهجن** | ⭐⭐⭐⭐ | ❓ غير مجرب |
| 7 | **أدوار المستخدمين** | `/admin/users/roles/list` | **BaseCrudPage** | ⚡ **مهجن** | ⭐⭐⭐⭐ | ❓ غير مجرب |
| 8 | **الملفات الشخصية** | `/admin/users/profiles/list` | **BaseCrudPage** | ⚡ **مهجن** | ⭐⭐⭐⭐ | ❓ غير مجرب |
| 9 | **المؤسسات** | `/admin/organizations/list` | **BaseCrudPage** | ⚡ **مهجن** | ⭐⭐⭐⭐⭐ | ⚠️ مشاكل |
| 10 | **الأدوية** | `/admin/medical/medications/list` | **BaseCrudPage** | ⚡ **مهجن** | ⭐⭐⭐⭐⭐ | ❌ مشاكل (65K) |
| 11 | **الخدمات الطبية** | `/admin/medicalservices/list` | **اتصال مباشر** | ❌ **قديم** | ⭐⭐ | ❓ غير مجرب |
| 12 | **التحاليل** | `/admin/labtests/list` | **اتصال مباشر** | ❌ **قديم** | ⭐⭐ | ❓ غير مجرب |
| 13 | **المعاملات المالية** | `/admin/pricing/transactions` | **اتصال مباشر** | ❌ **قديم** | ⭐⭐ | ❓ غير مجرب |
| 14 | **المحافظ** | `/admin/pricing/wallets` | **اتصال مباشر** | ❌ **قديم** | ⭐⭐ | ❓ غير مجرب |
| 15 | **فئات الخدمات** | `/admin/services/categories` | **اتصال مباشر** | ❌ **قديم** | ⭐⭐ | ❓ غير مجرب |
| 16 | **خيارات التسعير** | `/admin/services/pricing-options` | **اتصال مباشر** | ❌ **قديم** | ⭐⭐ | ❓ غير مجرب |
| 17 | **المنتجات** | `/admin/services/products` | **اتصال مباشر** | ❌ **قديم** | ⭐⭐ | ❓ غير مجرب |

### **🔍 شرح الحالات الثلاث:**

#### **1. ✅ النظام الجديد (BaseFactoryCrudPage)**
```yaml
المميزات:
  - يستخدم DbContextFactoryService
  - لا مشاكل Second operation
  - يدعم Excel بالكامل
  - أمثلة: Diseases, AdminMenuManager, PSPPrograms

الحالة: مثالي، يعمل 100%
```

#### **2. ⚡ النظام المهجن (BaseCrudPage المعدل)**
```yaml
المميزات:
  - BaseCrudPage الأصلي
  - لكن IGenericService مرتبط بـ DbContextFactory
  - يعمل بالنظام الجديد من الخلفية
  - أمثلة: Specialities, Organizations, Medications

الحالة: يعمل، لكن قد يحتاج تحسينات
```

#### **3. ❌ النظام القديم (اتصال مباشر)**
```yaml
المشاكل:
  - @inject BusinessDbContext Context مباشرة
  - عرضة لـ Second operation
  - لا يدعم Excel
  - أمثلة: MedicalServices, LabTests, Pricing

الحالة: يحتاج ترحيل عاجل
```

### **🛠️ خطوات الترحيل حسب النوع:**

#### **النوع أ: من ❌ قديم إلى ✅ جديد (الأولوية القصوى)**
```csharp
// الصفحات التي تستخدم Context مباشرة
@page "/admin/services/products"
@inject BusinessDbContext Context  // ❌ قديم

// التحويل:
// 1. إنشاء نموذج في Data/Models/
// 2. إضافة DbSet في BusinessDbContext
// 3. تسجيل IGenericService في Program.cs
// 4. تغيير الصفحة لـ BaseFactoryCrudPage
```

#### **النوع ب: من ⚡ مهجن إلى ✅ جديد (تحسين)**
```csharp
// الصفحات التي تستخدم BaseCrudPage
@inherits BaseCrudPage<Speciality>  // ⚡ مهجن

// التحويل (اختياري):
// 1. تغيير الوراثة لـ BaseFactoryCrudPage
// 2. إزالة أي injects غير ضرورية
// 3. تحديث عمليات CRUD لاستخدام DbFactory
```

### **🎯 أولويات الترحيل المحدثة:**

#### **🔴 أولوية عالية (الصفحات الحرجة):**
1. **MedicalServices** - خدمات طبية (مباشرة ← جديد)
2. **LabTests** - تحاليل (مباشرة ← جديد)  
3. **Organizations** - مؤسسات (مهجن ← جديد)
4. **Medications** - أدوية (مهجن ← جديد)

#### **🟡 أولوية متوسطة (تحسين):**
1. **UserProfiles** - ملفات شخصية
2. **Translations** - ترجمة
3. **Cities/Countries/Areas** - بيانات أساسية

#### **🟢 أولوية منخفضة (تعمل):**
1. **Specialities** - تعمل بشكل جيد
2. **PSP Programs** - مكتمل بالفعل
3. **Diseases** - مكتمل (نموذج مرجعي)

### **📊 حالة النظام الشاملة:**
```yaml
النظام الجديد: 3 صفحات (100% مكتمل)
النظام المهجن: 8 صفحات (تعمل، تحتاج تحسين)
النظام القديم: 8 صفحات (يحتاج ترحيل عاجل)
المجموع: 19 صفحة إدارية
```

### **⚡ الحقيقة الفنية المهمة:**
```csharp
// IGenericService الآن يستخدم DbContextFactory!
public class GenericService<T> : IGenericService<T> where T : class
{
    private readonly DbContextFactoryService _dbFactory;
    
    // جميع عمليات CRUD تستخدم:
    await _dbFactory.ExecuteWithNewContextAsync(async context =>
    {
        // ... عمليات قاعدة البيانات
    });
}

// النتيجة: حتى BaseCrudPage يعمل بالنظام الجديد!
// لأن IGenericService مرتبط بـ DbContextFactory
```

---

### ✅ **التصحيح النهائي:**

**الخلاصة:** لدينا **3 أنظمة تعمل بالتوازي**:
1. ✅ **الجديد**: BaseFactoryCrudPage (مثال: Diseases)
2. ⚡ **المهجن**: BaseCrudPage مع IGenericService المحدث (مثال: Specialities)  
3. ❌ **القديم**: اتصال مباشر بالـ Context (مثال: MedicalServices)

**الخطة:** ترحيل الصفحات من **❌ قديم** إلى **✅ جديد** أولاً، ثم تحسين **⚡ مهجن** إلى **✅ جديد** لاحقاً.

---
---

## 🗃️ **قاعدة البيانات للتنفيذ**

### **📊 جداول أساسية للتطوير:**
| الجدول | الأعمدة المهمة | العلاقات | حجم البيانات |
|--------|---------------|----------|--------------|
| **Medications** | MedicationID, MedicationNameAr, DrugFormID, MedicineCategoryID | ←DrugForms, ←MedicationCategories | 17,615 |
| **DrugForms** | DrugFormID, FormNameAr, FormCode | →Medications | 56 |
| **MedicationCategories** | CategoryID, CategoryNameAr, ParentCategoryID | ذاتية (هرمية) | 1,059 |
| **Organizations** | OrganizationID, OrganizationNameAr, OrganizationTypeID | →OrganizationTypes | 1,241 |
| **UserProfiles** | UserProfileID, UserID, FullNameAr | ←AspNetUsers | 5 |
| **SystemMenus** | MenuID, MenuNameAr, MenuType | →MenuItems | 3 |
| **MenuItems** | ItemID, TitleAr, Url, Icon, SystemMenuID | ←SystemMenus | 16 |

### **🔗 نماذج العلاقات في C#:**
```csharp
// مثال: Medication مع علاقات متعددة
public class Medication
{
    public int MedicationID { get; set; }
    public string MedicationNameAr { get; set; }
    
    // العلاقة مع DrugForms (الجديدة)
    public int? DrugFormID { get; set; }
    public virtual DrugForm DrugForm { get; set; }
    
    // العلاقة مع التصنيف الهرمي
    public int? MedicineCategoryID { get; set; }
    public virtual MedicationCategory MedicineCategory { get; set; }
    
    // العلاقة مع الشركات المنتجة
    public int? PharmaCompanyID { get; set; }
    public virtual Organization PharmaCompany { get; set; }
}
```

---

## 🚨 **نقاط التحقق قبل أي تنفيذ**

### **✅ قبل إنشاء صفحة جديدة:**
1. **الجدول موجود في DbContext؟** ← تحقق من BusinessDbContext.cs
2. **النموذج موجود في Data/Models؟** ← تحقق من وجود Model.cs
3. **الخدمة مسجلة في Program.cs؟** ← ابحث عن AddScoped
4. **المسار متاح في InteractiveMenu؟** ← تحقق من MenuItems

### **✅ قبل تعديل صفحة موجودة:**
1. **خذ نسخة احتياطية** من الملف
2. **تحقق من النمط الحالي** (BaseCrudPage أم BaseFactoryCrudPage)
3. **اختبر كل عملية CRUD** بعد التعديل
4. **تحقق من Excel** إذا كانت الصفحة تدعمه

### **✅ قبل إضافة مكون جديد:**
1. **هل يوجد مكون جاهز في Components/Shared/RubikCare؟**
2. **هل يحتاج تكوين خاص في Program.cs؟**
3. **هل يتوافق مع نظام الألوان الموحد؟**
4. **هل تم تجربته في صفحة Diseases؟**

---

## 📞 **مراجع سريعة أثناء العمل**

### **🔗 ملفات أساسية:**
```yaml
DbContext الرئيسي: 
  - Data/BusinessDbContext.cs
  - (يحتوي كل DbSets)

خدمة Factory: 
  - Data/Services/Core/DbContextFactoryService.cs
  - ⭐ أساسي لعدم Second operation

نظام القوائم:
  - Data/Services/Navigation/DynamicMenuService.cs
  - Pages/Layout/InteractiveMenu.razor

أمثلة ناجحة:
  - Pages/Admin/Medical/Diseases/Diseases.razor ⭐
  - Pages/Admin/PSP/Programs/PSPProgramEdit.razor
```

### **📞 عند مواجهة مشكلة:**
1. **تحقق من Project-Dashboard.md** (هذه الوثيقة)
2. **انسخ حل Diseases.razor** (نموذج ناجح)
3. **استخدم DbContextFactory** بدلاً من Context مباشر
4. **اختبر على عينة صغيرة** قبل التطبيق الكامل

---

<div align="center">

## 🎯 **ملخص للعمل السريع**

### **لإنشاء صفحة CRUD جديدة في 45 دقيقة:**
1. **انسخ Diseases.razor** كقالب
2. **عدل 5 نقاط أساسية** فقط
3. **سجل الخدمة** في Program.cs
4. **اختبر**: تحميل ← بحث ← إضافة ← Excel

### **للصفحات الحرجة التي تحتاج ترحيل:**
1. **Organizations.razor** (مؤسسات - 1,241 سجل)
2. **Medications.razor** (أدوية - 65K+ سجل)  
3. **UserProfiles.razor** (ملفات شخصية)

### **المكونات الجاهزة للاستخدام فوراً:**
- **RubikSmartTable** + **SearchBar** + **Pagination**
- **GenericModal** + **DataOperationsModal** (مع Excel)
- **RubikDropdown** + **RubikTabs** + **Rubiklistview**

</div>
```

---

# 📊 **Project Dashboard - RubikCare** (الإصدار 3.0)

## 🎯 **وثيقة الحالة الفعلية للمشروع - 14 فبراير 2026**

---

## 📋 **مقدمة: الغرض من هذه الوثيقة**
هذه وثيقة تقنية داخلية للمطور شادي ومساعده التقني، تعكس **الواقع الفعلي** للمشروع في هذه اللحظة، وتستخدم كمرجع سريع أثناء جلسات التطوير.

---

## 🏗️ **1. الهيكل الحقيقي للمشروع (RubikCare.sln)**

### **المشروعات الأربعة الفعلية:**
```yaml
📁 RubikCare.sln
├── 1. 📂 RubikCare.Domain (Class Library .NET 8)
│   ├── Entities/                    ← كل النماذج (Models)
│   │   ├── Disease.cs
│   │   ├── DiseaseType.cs
│   │   ├── Medication.cs
│   │   ├── DrugForm.cs
│   │   ├── MedicationCategory.cs
│   │   ├── Organization.cs
│   │   ├── OrganizationType.cs
│   │   ├── Resource.cs               ← 2,434 ترجمة
│   │   ├── ServiceCategory.cs         ← تم إنجازه بالكامل
│   │   ├── ServiceProduct.cs          ← تم إنجازه بالكامل
│   │   ├── ServiceTarget.cs           ← تم إنجازه بالكامل
│   │   ├── PSPProgram.cs
│   │   └── ... (باقي النماذج)
│   ├── Enums/
│   └── Constants/

├── 2. 📂 RubikCare.Persistence (.NET 8)
│   ├── Data/
│   │   ├── BusinessDbContext.cs      ← كل DbSets
│   │   ├── Configurations/            ← Fluent Configurations
│   │   └── Migrations/                 ← 40+ ملف
│   ├── Services/
│   │   ├── Core/
│   │   │   ├── DbContextFactoryService.cs ⭐ أساسي
│   │   │   └── GenericService.cs
│   │   ├── Localization/              ← نظام الترجمة ⭐
│   │   │   ├── TranslationStateService.cs
│   │   │   ├── LayoutDirectionService.cs
│   │   │   ├── LocalizationService.cs
│   │   │   └── ITranslationStateService.cs
│   │   ├── Sessions/                  ← ⭐ مهم جداً
│   │   │   └── UserSessionService.cs  ← التخزين المحلي
│   │   ├── Navigation/
│   │   │   └── DynamicMenuService.cs
│   │   └── PSP/
│   │       ├── IPSPProgramService.cs
│   │       └── PSPProgramService.cs
│   └── Repositories/                  ← (إذا وجدت)

├── 3. 📂 RubikCare.Shared.UI (Razor Class Library .NET 8)
│   ├── Components/
│   │   ├── Base/
│   │   │   ├── BaseFactoryCrudPage.cs
│   │   │   └── BaseFactoryCrudPageWithRelations.cs
│   │   ├── UI/                         ← المكونات المشتركة (13 مكون)
│   │   │   ├── GenericModal.razor
│   │   │   ├── DataOperationsModal.razor
│   │   │   ├── SearchBar.razor
│   │   │   ├── RubikSmartTable.razor
│   │   │   ├── Pagination.razor
│   │   │   ├── AlertMessage.razor
│   │   │   ├── RubikButton.razor
│   │   │   ├── RubikDropdown.razor
│   │   │   ├── RubikTabs.razor
│   │   │   ├── Rubiklistview.razor
│   │   │   ├── ImageUploader.razor
│   │   │   └── DynamicForm.razor
│   │   └── Localization/               ← ⭐ مكونات الترجمة هنا
│   │       ├── LocalizedText.razor
│   │       ├── LocalizedLabel.razor
│   │       └── LocalizedButton.razor
│   └── wwwroot/
│       ├── css/                         ← ملفات CSS (محدثة)
│       └── Assets/js/                   
│           ├── direction-protector.js
│           └── accessibility.js

├── 4. 📂 RubikCare.Web (Blazor Server .NET 8) ← المشروع الحالي
│   ├── Components/
│   │   ├── Layout/
│   │   │   ├── MainLayout.razor
│   │   │   ├── InteractiveMenu.razor
│   │   │   └── LanguageSwitcher.razor
│   │   ├── Pages/
│   │   │   ├── Admin/
│   │   │   │   ├── Medical/
│   │   │   │   │   ├── Diseases/
│   │   │   │   │   ├── Medications/
│   │   │   │   │   ├── Specialities/
│   │   │   │   │   └── LabTests/
│   │   │   │   ├── General/
│   │   │   │   │   ├── Countries/
│   │   │   │   │   ├── Cities/
│   │   │   │   │   └── Areas/
│   │   │   │   ├── Organizations/
│   │   │   │   ├── Users/
│   │   │   │   │   ├── Profiles/
│   │   │   │   │   └── Roles/
│   │   │   │   ├── PSP/
│   │   │   │   │   └── Programs/
│   │   │   │   └── Observability/      ← ← ⭐ تم تعديله من System
│   │   │   │       └── Observability.razor  ← ملف واحد فقط
│   │   │   └── Shared/
│   │   └── Features/                    ← للتنظيم المستقبلي
│   └── Program.cs

├── 5. 📂 RubikCare.Api (Minimal API .NET 8)
│   └── (قيد التطوير)

├── 6. 📂 RubikCare.Mobile (.NET MAUI)
│   └── (مخطط)

└── 📂 Documentation/
    ├── Project-Dashboard.md              ← هذه الوثيقة
    ├── Components-Guid.md                 ← دليل المكونات
    ├── Localizationplan.md                 ← نظام الترجمة
    ├── PSP-Business-wise.md                ← منطق أعمال PSP
    ├── PSP-Launch-Plan.md                  ← خطة إطلاق PSP
    ├── identity-guid.md                    ← دليل الهوية
    ├── Observability.md                     ← نظام المراقبة
    ├── TECHNICAL-IMPLEMENTATION-GUIDE.md    ← دليل التنفيذ
    └── RubikCare Solution Architecture v4.0.md
```

---

## ✅ **2. ما تم إنجازه 100%**

### **2.1 البنية التحتية Infrastructure**
| المكون | الحالة | ملاحظات |
|--------|--------|---------|
| **DbContextFactoryService** | ✅ مستقر | يحل مشكلة Second operation |
| **GenericService** | ✅ مستقر | يستخدم DbContextFactory |
| **UserSessionService** | ✅ مستقر | إدارة التخزين المحلي |

### **2.2 نظام الترجمة والاتجاه (جاهز تماماً)**
| المكون | الحالة | الوظيفة |
|--------|--------|---------|
| TranslationStateService | ✅ مستقر | مصدر الحقيقة للغة |
| LayoutDirectionService | ✅ مستقر | حساب الاتجاه |
| direction-protector.js | ✅ مستقر | يحمي الاتجاه |
| مكونات Localization | ✅ موجودة | LocalizedText, LocalizedLabel, LocalizedButton |

⚠️ **ملاحظة هامة:** نظام الترجمة **جاهز للاستخدام** ولكن **لم يتم تحويل أي صفحة** لاستخدامه بعد.

### **2.3 نظام الخدمات الهرمي (Service Categories) - ✅ مكتمل بالكامل**
```yaml
📦 ServiceCategories: تم إنجازه
  ├── نموذج ServiceCategory في Domain
  ├── DbSet في BusinessDbContext
  ├── علاقة ParentCategoryID (هرمية)
  └── شاشة الإدارة موجودة

📦 ServiceProducts: تم إنجازه
  ├── نموذج ServiceProduct
  ├── علاقة CategoryID ← ServiceCategories
  └── ServiceNature (FEATURE_BASED, SUBSCRIPTION_BASED, إلخ)

📦 ServiceTargets: تم إنجازه
  ├── نموذج ServiceTarget
  ├── TargetType (ORGANIZATION / USER_ROLE)
  └── TargetID (ربط مع OrganizationType أو Role)
```

### **2.4 المكونات المشتركة (13 مكون)**
جميع المكونات في `Components/Shared/UI` جاهزة ومستقرة.

### **2.5 نظام القوائم الديناميكي (InteractiveMenu)**
يعمل بشكل كامل مع:
- قائمة شخصية (USER_BASE)
- قوائم المؤسسات
- قائمة Admin لمؤسسة روبيك كير

---

## 🚧 **3. ما لم يتم إنجازه (Backlog) - مرتب حسب الأولوية**

### **🔴 أولوية قصوى (لأن المستخدم النهائي هو العميل)**

#### **3.1 شاشات برامج دعم المرضى (PSP) - العميل: شركات الأدوية**
| المهمة | الحالة | التفاصيل |
|--------|--------|----------|
| **مراجعة وتحسين شاشة تسجيل برامج PSP** | ⚠️ به مشاكل | إضافة/حذف البرامج يحتاج ضبط |
| **تصميم شاشة افتتاحية لشركات الأدوية** | ❌ لم تبدأ | واجهة مخصصة لتصميم البرامج |
| **إكمال منطق الأعمال PSP** | ⚠️ جزئي | ربط الخدمات بالبرامج |

#### **3.2 واجهات المستخدمين النهائيين - أولوية قصوى**
| المستخدم | المهمة | الحالة |
|----------|--------|--------|
| **الطبيب** | تصميم واجهة الطبيب | ❌ لم تبدأ |
| **الصيدلي** | تصميم واجهة الصيدلي | ❌ لم تبدأ |
| **المريض** | تصميم واجهة المريض | ❌ لم تبدأ |

#### **3.3 نظام الصلاحيات (Authorizations) - ⭐ مهم جداً**
| المهمة | الحالة | التأثير |
|--------|--------|---------|
| **شاشة صلاحيات الدخول** | ❌ لم تبدأ | مطلوبة لكل المنظمات |
| **فصل صلاحيات شركات الأدوية** | ❌ | عن العيادات عن الصيدليات |

#### **3.4 شاشات المستخدمين والإعدادات**
| المهمة | الحالة |
|--------|--------|
| **إصلاح شاشة UserProfile** | ⚠️ بها مشاكل |
| **شاشة تفضيلات المستخدم** | ❌ لم تبدأ |
| **إعدادات عامة للنظام (لكل مستخدم)** | ❌ لم تبدأ |

---

### **🟡 أولوية متوسطة (للمستخدمين الداخليين - موظفي روبيك كير)**

#### **3.5 ترجمة الصفحات (تحويل النصوص لاستخدام LocalizedText)**
| الصفحة | الحالة | عدد النصوص |
|--------|--------|------------|
| Diseases.razor | ⚠️ جزئي | 20-25 نص |
| Cities.razor | ❌ 0% | 15-20 نص |
| Areas.razor | ❌ 0% | 15-20 نص |
| Countries.razor | ❌ 0% | 15-20 نص |
| Specialities.razor | ❌ 0% | 20-25 نص |
| Medications.razor | ❌ 0% | 25-30 نص |
| Organizations.razor | ❌ 0% | 20-25 نص |
| UserProfiles.razor | ❌ 0% | 20-25 نص |
| Roles.razor | ❌ 0% | 15-20 نص |

**إجمالي النصوص المتبقية:** ~150-200 نص

---

### **🟢 أولوية منخفضة (تحسينات)**

#### **3.6 تحسينات عامة**
| المهمة | الحالة |
|--------|--------|
| **Dashboard احترافية للمشروع** | ❌ |
| **مراجعة شاملة لنمط الصفحات** | ⚠️ مستمرة |
| **توثيق API** | ❌ |

---

## 📊 **4. مصفوفة حالة الصفحات (حسب الجمهور)**

### **👥 صفحات الجمهور الخارجي (عملاء المنصة) - أولوية قصوى**

#### **أ. شركات الأدوية (Pharma Companies)**
| الصفحة | النظام الحالي | حالة الترجمة | حالة Excel |
|--------|--------------|--------------|------------|
| PSP Programs | ✅ جديد | ❌ | ✅ |
| Service Categories | ✅ مكتمل | ❌ | ❓ |
| Service Products | ✅ مكتمل | ❌ | ❓ |
| Service Targets | ✅ مكتمل | ❌ | ❓ |

#### **ب. الأطباء (Doctors)**
| الصفحة | الحالة | ملاحظات |
|--------|--------|---------|
| واجهة الطبيب | ❌ | لم تبدأ |
| إدارة المرضى | ❌ | لم تبدأ |
| وصف الأدوية | ❌ | لم تبدأ |

#### **ج. الصيادلة (Pharmacists)**
| الصفحة | الحالة |
|--------|--------|
| واجهة الصيدلي | ❌ |
| صرف الأدوية | ❌ |

#### **د. المرضى (Patients)**
| الصفحة | الحالة |
|--------|--------|
| واجهة المريض | ❌ |
| برامج الدعم | ❌ |
| المكافآت | ❌ |

### **👨‍💼 صفحات الجمهور الداخلي (موظفي روبيك كير) - أولوية أقل**

| الصفحة | النظام | حالة الترجمة | حالة Excel |
|--------|--------|--------------|------------|
| Diseases | ✅ جديد | ⚠️ جزئي | ✅ |
| Medications | ⚡ مهجن | ❌ | ❌ |
| Specialities | ⚡ مهجن | ❌ | ✅ |
| Cities | ⚡ مهجن | ❌ | ❓ |
| Countries | ⚡ مهجن | ❌ | ❓ |
| Areas | ⚡ مهجن | ❌ | ❓ |
| Organizations | ⚡ مهجن | ❌ | ⚠️ مشاكل |
| UserProfiles | ⚡ مهجن | ❌ | ❓ |
| Roles | ⚡ مهجن | ❌ | ❓ |
| Observability | جديد | ❌ | ❓ |

---

## 🧩 **5. المكونات المتوفرة في Shared.UI**

### **5.1 Components/UI (13 مكون)**
```
GenericModal.razor
DataOperationsModal.razor
SearchBar.razor
RubikSmartTable.razor
Pagination.razor
AlertMessage.razor
RubikButton.razor
RubikDropdown.razor
RubikTabs.razor
Rubiklistview.razor
ImageUploader.razor
DynamicForm.razor
ExcelService.cs
```

### **5.2 Components/Localization (3 مكونات جديدة)**
```
LocalizedText.razor     ← للنصوص
LocalizedLabel.razor    ← للتسميات
LocalizedButton.razor   ← للأزرار
```

---

## 🔧 **6. التصميم المعماري المقترح (للمستقبل)**

بناءً على واقع المشروع الحالي والتوسعات المخطط لها:

```yaml
📁 RubikCare.sln

├── 📂 RubikCare.Domain (NET 8)              ← Entities فقط
│   ├── Entities/
│   ├── Enums/
│   └── Constants/

├── 📂 RubikCare.Application (NET 8)          ← جديد مقترح
│   ├── Features/                              ← CQRS-like
│   │   ├── PSP/
│   │   ├── Users/
│   │   └── Medications/
│   ├── Common/
│   │   ├── Behaviors/
│   │   └── Exceptions/
│   └── Interfaces/                            ← Abstractions

├── 📂 RubikCare.Persistence (NET 8)          ← موجود
│   ├── Data/
│   ├── Services/
│   └── Migrations/

├── 📂 RubikCare.Shared.UI (RCL)              ← موجود
│   ├── Components/
│   │   ├── UI/
│   │   ├── Localization/
│   │   └── Features/                           ← تنظيم حسب الميزة
│   └── wwwroot/

├── 📂 RubikCare.Web (Blazor Server)          ← موجود
│   ├── Components/
│   │   ├── Layout/
│   │   └── Pages/
│   └── Infrastructure/                          ← Web-specific
│       ├── Services/
│       └── Middleware/

├── 📂 RubikCare.Api (Minimal API)             ← مخطط
│   ├── Endpoints/
│   └── Infrastructure/

└── 📂 RubikCare.Mobile (MAUI)                 ← مخطط
    ├── Features/
    └── Infrastructure/
```

### **💡 ملاحظات على التصميم:**
1. **Application** مفصول عن Persistence (لتوسعة API لاحقاً)
2. **Shared.UI** كمكتبة مستقلة (للاستخدام في Web و Mobile)
3. **Infrastructure** داخل كل مشروع للخدمات الخاصة بالمنصة

---

## 🎯 **7. خطة العمل للأيام القادمة (مرتبة حسب الأولوية)**

### **اليوم 1-2: شاشات PSP**
1. 🔧 إصلاح مشاكل شاشة إضافة/حذف برامج PSP
2. 🎨 تصميم شاشة افتتاحية لشركات الأدوية

### **اليوم 3-4: نظام الصلاحيات**
1. 🔐 تصميم شاشة صلاحيات الدخول
2. 🔧 إصلاح UserProfile

### **اليوم 5-6: واجهات المستخدمين**
1. 👨‍⚕️ بدء تصميم واجهة الطبيب
2. 💊 بدء تصميم واجهة الصيدلي
3. 🧑 بدء تصميم واجهة المريض

### **اليوم 7: تحسينات**
1. 📝 إضافة LocalizedText للصفحات (إذا تبقى وقت)
2. 📊 Dashboard احترافية

---

## ⚠️ **8. قواعد صارمة للتطوير**

1. **🚫 لا تعدل Migration يدوياً** - code-first فقط
2. **🚫 لا تستخدم DbContext مباشر** - استخدم DbContextFactory
3. **✅ استخدم GenericService** للـ CRUD
4. **✅ استخدم UserSessionService** للتخزين المحلي
5. **✅ استخدم LocalizedText** للنصوص (بعد التحويل)

---

## 📞 **9. مراجع سريعة للجلسات**

### **ملفات أساسية:**
```yaml
نموذج مرجعي: 
  - Diseases.razor (للنمط)

نظام الترجمة:
  - TranslationStateService.cs
  - LocalizedText.razor (في Shared.UI/Localization)

خدمة الجلسة:
  - UserSessionService.cs (في Persistence/Sessions)

نظام الخدمات:
  - ServiceCategory.cs, ServiceProduct.cs, ServiceTarget.cs (كلها مكتملة)
```

### **أدوات التصحيح:**
```javascript
// في Console
localStorage.getItem('RubikCare:Language')
document.documentElement.dir
```

---



## 📋 **دليل سريع للمطورين**

### **🔧 عند إنشاء صفحة جديدة:**
```mermaid
graph TD
    A[صفحة جديدة؟] --> B{نوع الصفحة}
    B -->|CRUD مع DbSet| C[BaseFactoryCrudPage]
    B -->|Identity| D[RoleManager/UserManager]
    B -->|علاقات معقدة| E[BaseFactoryCrudPageWithRelations]
    B -->|صفحة بسيطة| F[اتصال مباشر]
```

### **⚠️ قبل أي تعديل:**
1. ✅ تحقق من وجود DbSet في BusinessDbContext
2. ✅ تحقق من تسجيل الخدمة في Program.cs
3. ✅ استخدم DbContextFactory بدلاً من DbContext مباشر
4. ✅ تأكد من نظام code-first (لا تعديل يدوي في Migration)

---
